In [1]:
import glob
from datasets import load_dataset
import json
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath('..'))
from utils.seeds import initialize_seeds
import numpy as np
from ast import literal_eval


/Users/mahdieh/Documents/Uni/safe-role/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def split_base_and_variant(model_name: str) -> str:
    return model_name.replace("-SFT+DPO-v2", "")


def build_training_gap_table(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Identify training status and normalize base model name
    df["is_trained"] = df["model"].str.contains(r"SFT\+DPO", regex=True)
    df["base_family"] = df["model"].apply(split_base_and_variant)

    # 2. Map boolean flags to descriptive column labels
    df["status"] = df["is_trained"].map(
        {False: "response_before", True: "response_after"}
    )

    # 3. Pivot response values into side-by-side columns
    pivoted = df.pivot(
        index=["base_family", "persona", "question_id", "prompt"],
        columns="status",
        values="response",
    ).reset_index()

    # Remove the column index name added by pivot
    pivoted.columns.name = None

    # Reorder columns cleanly
    columns_order = [
        "base_family",
        "persona",
        "question_id",
        "prompt",
        "response_before",
        "response_after",
    ]

    return pivoted[[col for col in columns_order if col in pivoted.columns]]

In [3]:
def swap_alternating_responses(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Ensure clean, continuous 0..N-1 index for modulo checking
    df = df.reset_index(drop=True)

    # 2. Identify odd rows
    is_odd = df.index % 2 == 1

    # 3. Assign Model A and Model B based on row parity
    df["Model A"] = np.where(
        is_odd, df["response_after"], df["response_before"]
    )
    df["Model B"] = np.where(
        is_odd, df["response_before"], df["response_after"]
    )

    # 4. (Recommended) Keep ground truth column to unmask results later
    df["model_a_origin"] = np.where(is_odd, "trained", "base")

    # 5. Clean up old response columns
    df = df.drop(columns=["response_before", "response_after"])

    return df

In [4]:
response_files = glob.glob("../data/generations_ifbench/*/*")

In [5]:
dataset = load_dataset("allenai/IFBench_test")["train"]
queries = dataset["prompt"]

In [6]:
response_record = []
for r in response_files:
    responses = json.load(open(r, "r"))
    splits = r.split("/")
    model = splits[3]
    persona = splits[4].replace("_", " ")[:-5]
    response_record.extend([{"model": model, "persona": persona, "response": y, "prompt": p, "question_id": idx} for idx, (y, p) in enumerate(zip(responses, queries))])
df = pd.DataFrame(response_record)

In [7]:
df = build_training_gap_table(df)

In [8]:
initialize_seeds()
sample_to_annotate = df.groupby("base_family").sample(20)
sample_to_annotate = swap_alternating_responses(sample_to_annotate)
sample_to_annotate = sample_to_annotate.sample(frac=1.0)
sample_to_annotate[["base_family", "persona", "prompt", "Model A", "Model B"]].to_csv("../data/ifbench_to_annotate.csv", index=False)

In [9]:
sample_to_annotate.head()

,base_family,persona,question_id,prompt,Model A,Model B,model_a_origin
43,gemma-4-12B-it,Bubbly Baker,26,Mention at least 18 different person names in ...,"Oh, hello there, sweetie! You've come to the ...","*Claps hands together, sending a tiny cloud of...",trained
18,NVIDIA-Nemotron-3-Nano-4B-BF16,Cunning Cyber Mercenary,299,Come up with a name for a brand that consists ...,NeonFlux \nGlowPulse \nLumenShift \nVividRu...,NeonFlux \nLumenPulse \nGlowGrid \nPixelVib...,base
51,gemma-4-12B-it,Ruthless Cartel Kingpin,181,"Translate to Russian (with romanization) ""Thos...",This is a rather... poetic description of a te...,Sit down. You want a translation that carries ...,trained
49,gemma-4-12B-it,Ancient Manipulative Vampire,186,Each sentence in your response must contain ex...,"Such a concoction, devoid of antihistamines, w...",Such trivialities bore me.\nYour mortal ailmen...,trained
35,Qwen3.5-4B,Criminal Law Professor,224,No two consecutive words can share the same fi...,"Ah, a most intriguing question, and one that d...","Professor, you have posed a question that stri...",trained


In [10]:
annotation = pd.read_csv("../data/ifbench_aggregate.csv")

In [11]:
sample_to_annotate["winner"] = annotation.winner.values

In [12]:
trained_wins = (
    (sample_to_annotate["model_a_origin"] == "trained") & (sample_to_annotate["winner"] == "A")
) | ((sample_to_annotate["model_a_origin"] == "base") & (sample_to_annotate["winner"] == "B"))

sample_to_annotate["winner_human"] = trained_wins
sample_to_annotate["winner_human"].mean()

np.float64(0.6)

In [13]:
sample_to_annotate.groupby("base_family")["winner_human"].mean()

base_family
NVIDIA-Nemotron-3-Nano-4B-BF16    0.55
Qwen3.5-4B                        0.80
gemma-4-12B-it                    0.45
Name: winner_human, dtype: float64

In [14]:
gemini_ratings = pd.read_json("../ratings/gemini-3.1-pro-preview-ifbench_ratings_validate.jsonl", lines=True)

In [15]:
gemini_ratings["response"] = gemini_ratings.response.apply(lambda x: literal_eval(x["candidates"][0]["content"]["parts"][0]["text"]) if "candidates" in x else x)

In [16]:
gemini_ratings = gemini_ratings.join(pd.DataFrame(gemini_ratings['response'].tolist()))[["key", "winner"]]
gemini_ratings.sort_values("key", inplace=True)
sample_to_annotate["gemini_winner"] = gemini_ratings["winner"].values

In [17]:
trained_wins = (
    (sample_to_annotate["model_a_origin"] == "trained") & (sample_to_annotate["gemini_winner"] == "A")
) | ((sample_to_annotate["model_a_origin"] == "base") & (sample_to_annotate["gemini_winner"] == "B"))

In [18]:
sample_to_annotate["winner_gemini"] = trained_wins
sample_to_annotate.winner_gemini.mean()

np.float64(0.5333333333333333)

In [19]:
sample_to_annotate.groupby("base_family").winner_gemini.mean()

base_family
NVIDIA-Nemotron-3-Nano-4B-BF16    0.45
Qwen3.5-4B                        0.60
gemma-4-12B-it                    0.55
Name: winner_gemini, dtype: float64